# Phase 3 — re-train the validation-selected family winners (weights were lost)
The previous GPU session ended (Colab free-tier usage limit) before Google Drive finished uploading the trained weights.
This notebook re-trains ONLY the family winners chosen on validation, with their exact saved configurations, scores each
once on `test.csv` right after training, and calls `drive.flush_and_unmount()` at the end so every file is uploaded.
**Run all** on a GPU runtime and allow Google Drive. Model C: Skipped — no real conversation_id/message_order available.

In [ ]:
import sys, json, hashlib, zipfile, shutil
from pathlib import Path
from datetime import datetime
import torch
print("torch.cuda.is_available():", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("GPU runtime required: select an L4 / A100 / T4 runtime, then run all again.")
props = torch.cuda.get_device_properties(0)
print(f"GPU: {props.name} | VRAM {props.total_memory / 2**30:.1f} GiB | CUDA (torch build) {torch.version.cuda} | torch {torch.__version__}")
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
from google.colab import files
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    DRIVE = Path("/content/drive/MyDrive/NLP_sentiment")
    DRIVE.mkdir(parents=True, exist_ok=True)
    print("Google Drive enabled:", DRIVE)
except Exception as exc:
    DRIVE = Path("/content/NLP_sentiment")
    DRIVE.mkdir(parents=True, exist_ok=True)
    print("Google Drive unavailable; using local Colab storage:", exc)
PKG_ZIP = DRIVE / "transformer_package.zip"
if not PKG_ZIP.exists():
    print("Upload transformer_package.zip from D:/NLP/sentiment_app/artifacts/colab_package_phase3")
    up = files.upload()
    PKG_ZIP.write_bytes(up[next(iter(up))])
PKG = Path("/content/pkg"); shutil.rmtree(PKG, ignore_errors=True)
with zipfile.ZipFile(PKG_ZIP) as z: z.extractall(PKG)
manifest = json.loads((PKG / "MANIFEST.json").read_text())
bad = [n for n, h in manifest.items() if hashlib.sha256((PKG / n).read_bytes()).hexdigest() != h]
if bad: raise SystemExit(f"Package files do not match MANIFEST.json: {bad}")
print(f"All {len(manifest)} package files verified. Package: {(PKG / 'VERSION.txt').read_text() if (PKG / 'VERSION.txt').exists() else 'v1'}")

In [ ]:
!pip -q install datasets accelerate sentencepiece protobuf   # only installs what is missing

In [ ]:
sys.path.insert(0, str(PKG))
import colab_runner
RUNS2 = DRIVE / "phase3_rerun_winners"
runner = colab_runner.Runner(PKG / "data", RUNS2)
WINNERS = json.loads(r'''[
 {
  "experiment_id": "indicbert_lr2e-05_ce_ep3p2_bs16x1_wd0.01_wu0.1_len128",
  "model": "ai4bharat/IndicBERTv2-MLM-only",
  "learning_rate": 2e-05,
  "epochs_max": 3,
  "max_length": 128,
  "loss": "ce",
  "focal_gamma": null,
  "class_weights": null,
  "class_weight_scheme": null,
  "lang_tokens": false,
  "sampler": "random",
  "text_variant": "v1_current",
  "scheduler": "linear",
  "freeze_mode": null,
  "frozen_embeddings": false,
  "batch_size": 16,
  "grad_accum": 1,
  "effective_batch_size": 16,
  "weight_decay": 0.01,
  "warmup_ratio": 0.1,
  "label_smoothing": 0.0,
  "freeze_lower_layers_epoch1": 0,
  "early_stopping_patience": 2,
  "metric_for_best_model": "macro_f1",
  "seed": 42,
  "fp16": true,
  "no_fp16_flag": false,
  "group_by_length": true,
  "input_col": "cleaned_text",
  "input_prefix": "",
  "resumed_from_checkpoint": null,
  "input": "cleaned_text",
  "best_epoch": 3.0,
  "epochs_run": 3.0,
  "train_seconds": 1518.1,
  "best_checkpoint": "epoch 3 (/content/checkpoints/checkpoint-13227); intermediate checkpoints deleted, best weights kept per model family by colab_runner",
  "still_improving_at_last_epoch": true,
  "overfitting_signal": null,
  "trainer_best_val_macro_f1": 0.7504865594033877,
  "device": "Tesla T4",
  "split_sha256": {
   "train": "e0835f566132bda8355b32401c23026ca0037a450b245140f4d02bbf48ca3fcb",
   "validation": "427fe31b3b1873045c12423de5c9a6469968840a8506c452aef8f5e08e027d7e",
   "test": "935b3185c808c840e672654e770eeda4d70bfe1ac1c2fbd186a308c61eeef468"
  },
  "smoke_test": false,
  "test_used": false
 },
 {
  "experiment_id": "xlmr_lr1e-05_ce_ep3p2_bs16x1_wd0.01_wu0.1_len128",
  "model": "xlm-roberta-base",
  "learning_rate": 1e-05,
  "epochs_max": 3,
  "max_length": 128,
  "loss": "ce",
  "focal_gamma": null,
  "class_weights": null,
  "class_weight_scheme": null,
  "lang_tokens": false,
  "sampler": "random",
  "text_variant": "v1_current",
  "scheduler": "linear",
  "freeze_mode": null,
  "frozen_embeddings": false,
  "batch_size": 16,
  "grad_accum": 1,
  "effective_batch_size": 16,
  "weight_decay": 0.01,
  "warmup_ratio": 0.1,
  "label_smoothing": 0.0,
  "freeze_lower_layers_epoch1": 0,
  "early_stopping_patience": 2,
  "metric_for_best_model": "macro_f1",
  "seed": 42,
  "fp16": true,
  "no_fp16_flag": false,
  "group_by_length": true,
  "input_col": "cleaned_text",
  "input_prefix": "",
  "resumed_from_checkpoint": null,
  "input": "cleaned_text",
  "best_epoch": 2.0,
  "epochs_run": 3.0,
  "train_seconds": 1487.3,
  "best_checkpoint": "epoch 2 (/content/checkpoints/checkpoint-8818); intermediate checkpoints deleted, best weights kept per model family by colab_runner",
  "still_improving_at_last_epoch": false,
  "overfitting_signal": null,
  "trainer_best_val_macro_f1": 0.7487087749347326,
  "device": "Tesla T4",
  "split_sha256": {
   "train": "e0835f566132bda8355b32401c23026ca0037a450b245140f4d02bbf48ca3fcb",
   "validation": "427fe31b3b1873045c12423de5c9a6469968840a8506c452aef8f5e08e027d7e",
   "test": "935b3185c808c840e672654e770eeda4d70bfe1ac1c2fbd186a308c61eeef468"
  },
  "smoke_test": false,
  "test_used": false
 },
 {
  "experiment_id": "muril_lr3e-05_ce_ep3p2_bs16x1_wd0.01_wu0.1_len128",
  "model": "google/muril-base-cased",
  "learning_rate": 3e-05,
  "epochs_max": 3,
  "max_length": 128,
  "loss": "ce",
  "focal_gamma": null,
  "class_weights": null,
  "class_weight_scheme": null,
  "lang_tokens": false,
  "sampler": "random",
  "text_variant": "v1_current",
  "scheduler": "linear",
  "freeze_mode": null,
  "frozen_embeddings": false,
  "batch_size": 16,
  "grad_accum": 1,
  "effective_batch_size": 16,
  "weight_decay": 0.01,
  "warmup_ratio": 0.1,
  "label_smoothing": 0.0,
  "freeze_lower_layers_epoch1": 0,
  "early_stopping_patience": 2,
  "metric_for_best_model": "macro_f1",
  "seed": 42,
  "fp16": true,
  "no_fp16_flag": false,
  "group_by_length": true,
  "input_col": "cleaned_text",
  "input_prefix": "",
  "resumed_from_checkpoint": null,
  "input": "cleaned_text",
  "best_epoch": 3.0,
  "epochs_run": 3.0,
  "train_seconds": 1361.5,
  "best_checkpoint": "epoch 3 (/content/checkpoints/checkpoint-13227); intermediate checkpoints deleted, best weights kept per model family by colab_runner",
  "still_improving_at_last_epoch": true,
  "overfitting_signal": null,
  "trainer_best_val_macro_f1": 0.741375592036874,
  "device": "Tesla T4",
  "split_sha256": {
   "train": "e0835f566132bda8355b32401c23026ca0037a450b245140f4d02bbf48ca3fcb",
   "validation": "427fe31b3b1873045c12423de5c9a6469968840a8506c452aef8f5e08e027d7e",
   "test": "935b3185c808c840e672654e770eeda4d70bfe1ac1c2fbd186a308c61eeef468"
  },
  "smoke_test": false,
  "test_used": false
 }
]''')

def export(tag):
    z_path = DRIVE / "phase3_results_rerun.zip"
    with zipfile.ZipFile(z_path, "w", zipfile.ZIP_DEFLATED) as z:
        for p in sorted(RUNS2.rglob("*")):
            if p.is_file() and not any(part.startswith("_") for part in p.relative_to(RUNS2).parts):
                z.write(p, p.relative_to(RUNS2).as_posix())
        if (RUNS2 / "_selected_for_test.json").exists():
            z.write(RUNS2 / "_selected_for_test.json", "_selected_for_test.json")
    print("EXPORTED", tag, z_path, f"({z_path.stat().st_size / 1e6:.1f} MB)")

for saved in WINNERS:
    cfg = colab_runner.cfg_from_saved(saved)
    eid = colab_runner.exp_id(dict(colab_runner.DEFAULT, **cfg))
    assert eid == saved["experiment_id"], (eid, saved["experiment_id"])
    fam = cfg["family"]
    print("=== re-training", fam, eid)
    runner.run(cfg)
    runner.final_test()                      # tests the best (= only) run of each family present, once
    t = json.loads((RUNS2 / eid / "test_metrics.json").read_text())
    print(f"{fam}: {eid} TEST macro F1 {t['macro_f1']:.4f} acc {t['accuracy']:.4f}")
    best = RUNS2 / f"_best_model_{fam}" / "model"
    mz = DRIVE / f"phase3_model_{fam}.zip"
    with zipfile.ZipFile(mz, "w", zipfile.ZIP_STORED) as z:
        for p in sorted(best.iterdir()):
            if p.is_file(): z.write(p, p.name)
    print("WEIGHTS", fam, mz, f"({mz.stat().st_size / 1e9:.2f} GB)")
    export(fam)

In [ ]:
# Wait until Google Drive has really uploaded everything (the mount writes to a local cache first).
from google.colab import drive
drive.flush_and_unmount()
print("DRIVE_FLUSHED: all files uploaded")